In [1]:
# %% [markdown]
# # 10_bird_phase2_v2_experiment.ipynb
# Runs the v2 architecture against the ORIGINAL 106-question BIRD
# financial benchmark (Phase 2), not the 60-question credit rating
# benchmark used throughout Sections 4-5.
#
# Purpose: partially addresses Reviewer 2 Concern #3 ("Phases 2 and 3
# share the same underlying BIRD financial database schema...
# generalizability to proprietary credit rating schemas with
# different encoding conventions is unaddressed"). This does NOT
# test a genuinely different schema (that remains a real limitation,
# noted below), but DOES test the v2 architecture against a
# genuinely different, externally-authored question set on the SAME
# schema -- the BIRD financial benchmark's 106 questions were written
# by the BIRD benchmark authors, entirely independent of this paper's
# 60-question credit rating benchmark and its YAML concept design.
#
# IMPORTANT CAVEAT (documented in the manuscript, not just here):
# financial_semantic_layer.yaml's concept vocabulary was originally
# designed against the 60-question credit rating benchmark. Some BIRD
# Phase 2 questions may reference concepts, tables (e.g. 'order'), or
# query shapes not covered by the current YAML -- this is EXPECTED
# and should be reported as a finding (coverage gaps under a
# genuinely external question set), not treated as a bug to silently
# work around. Section 2 below runs a coverage check for exactly this
# reason before spending any API budget.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import os
import re
import pandas as pd
import time
import yaml
from utils import query_semantic_layer_v2, evaluate, DB_PATH

print(f"DB_PATH in use: {DB_PATH}")

# %% [markdown]
# ## 1. Load the ORIGINAL BIRD Phase 2 questions (106, financial
#     domain only) -- NOT the 60-question credit rating benchmark

# %%
candidate_json_paths = [
    "./dev/dev_20240627/dev.json",
    "./dev/dev.json",
    "./dev/dev_20240627/dev_databases/dev.json",
]

found_json = None
for p in candidate_json_paths:
    if os.path.exists(p):
        found_json = p
        break

if not found_json:
    raise FileNotFoundError(
        "Could not locate dev.json (BIRD's question metadata file) at any "
        f"of: {candidate_json_paths}. This is the same file "
        "01_fetch_bird_data.py located during initial setup -- if that "
        "notebook ran successfully before, dev.json should be present."
    )

with open(found_json, 'r', encoding='utf-8') as f:
    all_bird_questions = json.load(f)

bird_questions = [q for q in all_bird_questions if q.get('db_id') == 'financial']
print(f"Loaded {len(bird_questions)} BIRD financial-domain questions from {found_json}")
if len(bird_questions) != 106:
    print(f"!! WARNING: expected 106 questions per the manuscript, found {len(bird_questions)}.")

df_bird = pd.DataFrame(bird_questions)
print(f"\nFields available: {sorted(df_bird.columns)}")

if 'difficulty' not in df_bird.columns:
    df_bird['difficulty'] = 'unlabeled'
    print("\nNOTE: BIRD's dev.json does not provide difficulty labels")
    print("(unlike this paper's 60-question benchmark). Assigned")
    print("'unlabeled' as a placeholder -- do NOT report a difficulty")
    print("breakdown for this experiment without a real annotation source.")

if 'category' not in df_bird.columns:
    df_bird['category'] = 'unlabeled'

# %% [markdown]
# ## 2. Coverage check -- BEFORE spending any API budget, verify how
#     many of these 106 EXTERNALLY-AUTHORED questions the current
#     YAML concept vocabulary can even express. This is itself part
#     of the generalization finding, not a preprocessing step to hide.

# %%
with open('financial_semantic_layer.yaml') as f:
    sl_yaml = yaml.safe_load(f)

known_tables = {cube['sql_table'] for cube in sl_yaml['cubes']}

def references_uncovered_table(sql):
    # Match both bare identifiers (FROM account) and quoted/backticked
    # identifiers (FROM "order", FROM `order`) -- SQL commonly requires
    # quoting for reserved-word table names like 'order', and the BIRD
    # financial schema's order table is exactly this case. The original
    # \w+-only pattern silently missed quoted references entirely.
    pattern = r'\bFROM\s+["`]?(\w+)["`]?|\bJOIN\s+["`]?(\w+)["`]?'
    tables_in_sql = set(re.findall(pattern, sql, re.IGNORECASE))
    tables_in_sql = {t for pair in tables_in_sql for t in pair if t}
    uncovered = tables_in_sql - known_tables
    return uncovered

coverage_issues = []
for idx, row in df_bird.iterrows():
    uncovered = references_uncovered_table(row['SQL'])
    if uncovered:
        coverage_issues.append((idx, uncovered))

print(f"\n{len(coverage_issues)} / {len(df_bird)} BIRD questions reference a table with")
print("NO defined cube in financial_semantic_layer.yaml:")
for idx, tables in coverage_issues[:20]:
    print(f"  [row {idx}]: references {tables}")
if len(coverage_issues) > 20:
    print(f"  ... and {len(coverage_issues) - 20} more")

print(f"\nThese questions are EXPECTED to fail (compile_error, since the")
print(f"LLM cannot reference a concept from a cube that doesn't exist).")
print(f"This is reported as a genuine coverage-gap finding under external")
print(f"question authorship, not filtered out before running.")

# %% [markdown]
# ## 3. Run the full experiment (106 questions x 5 iterations = 530
#     API calls)

# %%
N_ITER = 5
RESULTS_PATH = './results_phase2_bird_v2_arch.csv'

if 'question_id' not in df_bird.columns:
    df_bird = df_bird.reset_index(drop=True)
    df_bird['question_id'] = ['BIRD_F' + str(i).zfill(3) for i in df_bird.index]

total = len(df_bird)
print(f"\nTotal questions: {total}")
print(f"Iterations per question: {N_ITER}")
print(f"Total API calls if run from scratch: {total * N_ITER}")

try:
    df_saved = pd.read_csv(RESULTS_PATH)
    results = df_saved.to_dict('records')
    done_ids = set(
        q_id for q_id, grp in df_saved.groupby('question_id')
        if grp['iteration'].nunique() >= N_ITER
    )
    print(f"Resuming — completed: {len(done_ids)} / {total}")
except FileNotFoundError:
    results = []
    done_ids = set()
    print("Starting fresh.")

print("\n" + "=" * 70)
print("RUNNING BIRD PHASE 2 v2 EXPERIMENT")
print("=" * 70)

for i, row in df_bird.iterrows():
    q_id = row['question_id']
    question = row['question']
    evidence = row.get('evidence', '')
    gold_sql = row['SQL']
    difficulty = row['difficulty']
    category = row['category']

    if q_id in done_ids:
        print(f"[{i+1:3d}/{total}] {q_id} — skipped (done)")
        continue

    iter_correct = 0
    iter_compile_err = 0
    iter_exec_err = 0

    for iteration in range(N_ITER):
        try:
            res = query_semantic_layer_v2(question, evidence)
        except Exception as e:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': None, 'execution_error': f"API call failed: {e}",
                'concept_sql': None, 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': None, 'prompt_chars': None,
                'substitutions': None,
            })
            iter_exec_err += 1
            continue

        if res['compile_error']:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': res['compile_error'], 'execution_error': None,
                'concept_sql': res['concept_sql'], 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': res['latency_ms'],
                'prompt_chars': res['prompt_chars'],
                'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
            })
            iter_compile_err += 1
            continue

        ev = evaluate(res['sql'], gold_sql)
        correct = bool(ev['is_correct']) if ev['is_correct'] is not None else False
        if correct:
            iter_correct += 1
        if ev.get('error'):
            iter_exec_err += 1

        results.append({
            'question_id': q_id, 'difficulty': difficulty, 'category': category,
            'iteration': iteration, 'is_correct': correct,
            'compile_error': None, 'execution_error': ev.get('error'),
            'concept_sql': res['concept_sql'], 'compiled_sql': res['sql'], 'gold_sql': gold_sql,
            'question': question, 'latency_ms': res['latency_ms'],
            'prompt_chars': res['prompt_chars'],
            'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
        })

        time.sleep(0.3)

    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False, encoding='utf-8-sig')

    df_tmp = pd.DataFrame(results)
    running_acc = df_tmp['is_correct'].mean()
    print(f"[{i+1:3d}/{total}] {q_id} correct={iter_correct}/{N_ITER} "
          f"compile_err={iter_compile_err}/{N_ITER} exec_err={iter_exec_err}/{N_ITER} "
          f"| Running accuracy: {running_acc:.1%}")

print("=" * 70)
print(f"Experiment complete. Saved: {RESULTS_PATH}")
print("=" * 70)

# %% [markdown]
# ## 4. Final summary

# %%
df_final = pd.read_csv(RESULTS_PATH)

print(f"Total records: {len(df_final)}")
print(f"Questions: {df_final['question_id'].nunique()}")

print(f"\nOverall accuracy (v2, BIRD Phase 2 original 106 questions): "
      f"{df_final['is_correct'].mean():.1%}")

n_compile_err = df_final['compile_error'].notna().sum()
n_exec_err = df_final['execution_error'].notna().sum()
n_correct = df_final['is_correct'].sum()
n_wrong_result = len(df_final) - n_compile_err - n_exec_err - n_correct

print(f"\n--- Two-stage failure taxonomy ---")
print(f"  Correct         : {n_correct} ({n_correct/len(df_final):.1%})")
print(f"  Compile errors  : {n_compile_err} ({n_compile_err/len(df_final):.1%})")
print(f"  Execution errors: {n_exec_err} ({n_exec_err/len(df_final):.1%})")
print(f"  Wrong result    : {n_wrong_result} ({n_wrong_result/len(df_final):.1%})")

print("""

NEXT STEP: compare this BIRD Phase 2 v2 accuracy against Phase 2's
original v1 results (results_phase2_bird_v2.csv, if available) using
the same question-level paired methodology as
06_statistical_reanalysis.py, and report both figures plus the
coverage-gap finding in a manuscript subsection addressing Reviewer 2
Concern #3. Frame this explicitly as "same schema, externally-
authored question set" -- NOT as a test of a genuinely different
schema, which remains a real limitation to disclose alongside this
result.
""")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Loaded 106 BIRD financial-domain questions from ./dev/dev_20240627/dev.json

Fields available: ['SQL', 'db_id', 'difficulty', 'evidence', 'question', 'question_id']

4 / 106 BIRD questions reference a table with
NO defined cube in financial_semantic_layer.yaml:
  [row 69]: references {'order'}
  [row 74]: references {'District', 'Loan', 'Account'}
  [row 75]: references {'order'}
  [row 84]: references {'order'}

These questions are EXPECTED to fail (compile_error, since the
LLM cannot reference a concept from a cube that doesn't exist).
This is reported as a genuine coverage-gap finding under external
question authorship, not filtered out before running.

Total questions: 106
Iterations per question: 5
Total API calls if run from scratch: 530
Starting fresh.

RUNNING BIRD PHASE 2 v2 EXPERIMENT
[  1/106] 89 correct=0/5 compile_err=1/5 exec_err=0/5 | Running accuracy: 

In [3]:
# %% [markdown]
# # DIAGNOSTIC 2: deep dive on the measure-in-WHERE dominant failure
# Two checks:
#   1. Question text integrity -- confirm BIRD questions were parsed
#      correctly from dev.json (no truncation, no field misalignment)
#   2. Sample actual measure-in-WHERE failures to see whether BIRD's
#      question phrasing systematically differs from the credit
#      rating benchmark's in a way that explains the pattern

# %%
import pandas as pd
import json
import os
pd.set_option('display.max_colwidth', 500)

df = pd.read_csv('results_phase2_bird_v2_arch.csv')

# %% [markdown]
# ## 1. Question text integrity check

# %%
candidate_json_paths = [
    "./dev/dev_20240627/dev.json",
    "./dev/dev.json",
    "./dev/dev_20240627/dev_databases/dev.json",
]
found_json = next((p for p in candidate_json_paths if os.path.exists(p)), None)

if found_json:
    with open(found_json, 'r', encoding='utf-8') as f:
        all_bird_questions = json.load(f)
    bird_financial = [q for q in all_bird_questions if q.get('db_id') == 'financial']

    print(f"Original dev.json financial questions: {len(bird_financial)}")
    print("\nFirst 5 ORIGINAL question texts (directly from dev.json, unmodified):")
    for q in bird_financial[:5]:
        print(f"\n  question: {q.get('question', 'MISSING')!r}")
        print(f"  evidence: {q.get('evidence', 'MISSING')!r}")
        print(f"  SQL: {q.get('SQL', 'MISSING')!r}")

    print("\n\nSame first 5, as STORED in results_phase2_bird_v2_arch.csv:")
    stored_questions = df.drop_duplicates('question_id').sort_values('question_id').head(5)
    for _, row in stored_questions.iterrows():
        print(f"\n  question_id: {row['question_id']}")
        print(f"  question (stored): {row['question']!r}")
else:
    print("!! dev.json not found -- cannot cross-check against original source.")

# %% [markdown]
# ## 2. Check for obviously malformed question text across ALL 106
#     questions

# %%
unique_questions = df.drop_duplicates('question_id')[['question_id', 'question']]
print(f"Scanning {len(unique_questions)} unique questions for signs of malformed text...")

suspicious = []
for _, row in unique_questions.iterrows():
    q = str(row['question'])
    flags = []
    if len(q) < 15:
        flags.append('unusually short')
    if 'XXXX' in q.upper() or '????' in q or '____' in q:
        flags.append('placeholder-like pattern')
    if q.count('  ') > 3:
        flags.append('multiple double-spaces')
    if flags:
        suspicious.append((row['question_id'], q, flags))

print(f"\n{len(suspicious)} questions flagged as potentially malformed:")
for qid, q, flags in suspicious[:20]:
    print(f"\n  [{qid}] flags={flags}")
    print(f"    {q}")

# %% [markdown]
# ## 3. Sample actual measure-in-WHERE failures

# %%
measure_where_failures = df[
    df['compile_error'].str.contains('WHERE clause', na=False)
]
print(f"{len(measure_where_failures)} measure-in-WHERE failures total.")
print("\nSample of 10 DISTINCT questions with this failure:")

seen_q = set()
count = 0
for _, row in measure_where_failures.iterrows():
    if row['question_id'] in seen_q:
        continue
    seen_q.add(row['question_id'])
    count += 1
    if count > 10:
        break
    print(f"\n--- [{row['question_id']}] ---")
    print(f"  question: {row['question']}")
    print(f"  concept_sql: {row['concept_sql']}")
    print(f"  error: {row['compile_error'][:200]}")

# %% [markdown]
# ## 4. Compare question PHRASING STYLE: BIRD vs credit-rating
#     benchmark, specifically around aggregate/filter language

# %%
print("\n" + "=" * 70)
print("PHRASING PATTERN CHECK")
print("=" * 70)
print("""
Look for BIRD questions phrased as simple existence/count filters
without explicit grouping -- if BIRD's 106 questions skew heavily
toward this simple, ungrouped phrasing pattern relative to the
credit-rating benchmark's stratified 60, that distributional
difference alone could explain a much higher measure-in-WHERE rate.
""")

ungrouped_count = 0
grouped_signal_words = ['each', 'by', 'group', 'per', 'category', 'top']
for _, row in unique_questions.iterrows():
    q = str(row['question']).lower()
    has_grouping_signal = any(w in q for w in grouped_signal_words)
    if not has_grouping_signal:
        ungrouped_count += 1

print(f"BIRD questions with no grouping-signal words ('each','by','per', etc.): "
      f"{ungrouped_count} / {len(unique_questions)} ({ungrouped_count/len(unique_questions):.1%})")
print("(Rough heuristic -- use it to gauge distributional skew, then verify")
print(" against the actual samples printed above.)")

Original dev.json financial questions: 106

First 5 ORIGINAL question texts (directly from dev.json, unmodified):

  question: 'How many accounts who choose issuance after transaction are staying in East Bohemia region?'
  evidence: "A3 contains the data of region; 'POPLATEK PO OBRATU' represents for 'issuance after transaction'."
  SQL: "SELECT COUNT(T2.account_id) FROM district AS T1 INNER JOIN account AS T2 ON T1.district_id = T2.district_id WHERE T1.A3 = 'east Bohemia' AND T2.frequency = 'POPLATEK PO OBRATU'"

  question: 'How many accounts who have region in Prague are eligible for loans?'
  evidence: 'A3 contains the data of region'
  SQL: "SELECT COUNT(T1.account_id) FROM account AS T1 INNER JOIN loan AS T2 ON T1.account_id = T2.account_id INNER JOIN district AS T3 ON T1.district_id = T3.district_id WHERE T3.A3 = 'Prague'"

  question: 'The average unemployment ratio of 1995 and 1996, which one has higher percentage?'
  evidence: 'A12 refers to unemploymant rate 1995; A13 refers